# 드론 풋볼 영상 분석 파이프라인 v7: 실행 노트북

영상 → **경기장 영역(ROI) 자동 계산 (SIFT + 호모그래피) + 드론이 움직이면 바로 다시 계산 (6b)** → **파인튜닝 YOLO26m 탐지 + 팀 분류 한 번에 (5b)**
→ **ByteTrack 추적 (2a 튜닝 설정)** → **앞뒤 15프레임 팀 라벨 + 팀이 바뀌면 트랙 자르기 (6a)** → `tracking_final.json` + 확인용 영상

| 항목 | 내용 |
|---|---|
| 입력 | Drive 폴더의 드론 영상 (ROI는 **자동**: 같은 경기장의 참조 프레임과 호모그래피로 계산) |
| 출력 | `pipeline_outputs/v7/{영상이름}_tracking_final.json` (v3와 같은 형식) · `{영상이름}_v7.mp4` |
| 필요 파일 | `eval/finetune/yolo26_ft_best.pt` (5b) · `eval/tracker_config_v4.json` (2a) |
| 속도 | T4 기준 약 0.2초/프레임 → 1분짜리(1800프레임) 클립에 약 6~7분 |

**v6 → v7에서 바뀐 것**

| | v6 | v7 | 근거 |
|---|---|---|---|
| 팀 라벨 | 트랙 전체 다수결 | 매 프레임 앞뒤 15프레임 다수결 + 팀이 바뀌면 새 ID | 6a: 팀 정확도 0.922 → **0.992**, 홍익↔국민 오류 36 → **3** |
| ROI 갱신 | 30프레임마다 | 30프레임마다 + 5프레임마다 화면 이동 검사, 밀렸으면 바로 재계산 | 6b: ROI IoU 최저 0.860 → **0.991**, 0144 인원 스파이크 35 → **14** |

**평가 결과 (평가 클립 2개, 20프레임 정답)**: 탐지 F1 0.955 · 팀 정확도 0.992 · 심판 Recall 1.00 · 득점자를 처음부터 끝까지 ID 하나로 추적

**사용 범위**: 탐지 모델은 *2025년 홍익 vs 국민 경기*의 유니폼으로 파인튜닝했어요. 다른 경기는 5a·5b로 그 경기 프레임을 라벨링해 다시 학습하세요.

**런타임: T4 GPU** · 위에서부터 순서대로 실행

In [ ]:
!pip install -q ultralytics
import os, json, time, shutil
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from google.colab import drive
drive.mount('/content/drive')

# 한글 글꼴 (그래프의 한글이 네모로 깨지지 않게)
import subprocess, matplotlib.font_manager as fm
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
if os.path.exists(_font):
    fm.fontManager.addfont(_font); plt.rcParams["font.family"] = "NanumGothic"
else:
    print("⚠️ 한글 글꼴 설치 실패: 그래프의 한글이 네모로 보일 수 있어요")
plt.rcParams["axes.unicode_minus"] = False

## ① 설정

`CLIPS`에 처리할 영상 이름(확장자 없이)을 넣으세요.
- **ROI를 `None`으로 두면 자동 계산**: 좌표를 직접 정해둔 클립(참조)과 SIFT 특징점을 짝지어 호모그래피로 ROI를 옮겨요.
- 드론이 움직이면: 5프레임마다 화면 이동을 빠르게 재서(1/4 해상도 ORB), ROI가 3% 이상 밀렸으면 그 자리에서 다시 계산해요. 움직임이 없어도 30프레임마다 한 번은 다시 계산해요.
- 아래 4개는 지금까지 직접 정한 ROI예요. 이 4개가 **참조**가 돼요 (경기장 양쪽 끝 2곳씩).
- 자동 ROI가 실패하면(경기장의 다른 곳을 찍은 영상) ②에서 격자가 그려진 첫 프레임이 나와요. 꼭짓점을 직접 넣으면 그 클립도 **새 참조**가 돼요.

In [ ]:
FOLDER_PATH      = "/content/drive/MyDrive/2025년 홍익 vs 국민 드론뷰"
DRIVE_OUTPUT_DIR = os.path.join(FOLDER_PATH, "pipeline_outputs")
EVAL_DIR         = os.path.join(DRIVE_OUTPUT_DIR, "eval")
OUT_DIR          = os.path.join(DRIVE_OUTPUT_DIR, "v7")
os.makedirs(OUT_DIR, exist_ok=True)

CLIPS = {
    # 영상 이름: ROI 꼭짓점 [[x, y], ...]  (None이면 자동 ROI)
    "DJI_20250913133612_0144_D": [[245, 530], [1860, 543], [1919, 776], [1919, 1079], [107, 1079]],
    "DJI_20250913135020_0156_D": [[0, 0], [1632, 0], [1712, 810], [0, 787]],
    "DJI_20250913133530_0143_D": [[270, 459], [1880, 469], [1919, 612], [1919, 1079], [119, 1079]],
    "DJI_20250913135111_0157_D": [[0, 0], [1675, 0], [1761, 737], [0, 733]],
    # "DJI_새영상이름_D": None,      ← 새 영상은 이렇게 None (자동 ROI)
}
REF_FRAME = {"DJI_20250913133612_0144_D": 413, "DJI_20250913135020_0156_D": 167}   # 직접 ROI를 정한 프레임 (없으면 1)
ROI_MARGIN_PX = 5            # ROI를 바깥으로 넓히는 여유 (1a)
ROI_UPDATE_EVERY = 30        # 움직임이 없어도 30프레임마다 ROI 다시 계산
ROI_FOLLOW = True            # True: 직접 ROI를 정한 클립도 드론이 움직이면 ROI가 따라감
MOTION_EVERY = 5             # 6b: 5프레임마다 화면 이동 검사
MOTION_SCALE = 0.25          # 6b: 이동 검사는 1/4 해상도
MOTION_IOU_MIN = 0.97        # 6b: 이동만큼 옮긴 ROI가 원래 ROI와 97%보다 덜 겹치면 다시 계산

# 모델 · 설정 (실험에서 정한 값 그대로)
DET_WEIGHTS = os.path.join(EVAL_DIR, "finetune", "yolo26_ft_best.pt")   # 5b 파인튜닝 (홍익 · 국민 · 심판)
DET_IMGSZ, DET_CONF = 1920, 0.05
TRACKER_CFG_PATH = os.path.join(EVAL_DIR, "tracker_config_v4.json")
TEAM_WINDOW = 15             # 6a: 앞뒤 15프레임 다수결
SPLIT_ON_TEAM_CHANGE = True  # 6a: 팀 라벨이 바뀌면 새 ID (한 ID = 한 사람 = 한 팀)

MAKE_VIDEO = True            # 확인용 영상(박스·팀·ID 표시) 만들기
OVERWRITE = False            # False면 이미 결과 JSON이 있는 클립은 건너뜀 (중간에 끊겨도 이어서 실행)

for p in (DET_WEIGHTS, TRACKER_CFG_PATH):
    assert os.path.exists(p), f"필요한 파일이 없습니다: {p}"


def video_path_of(stem):
    for ext in (".MP4", ".mp4", ".MOV", ".mov"):
        p = os.path.join(FOLDER_PATH, stem + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"영상을 찾을 수 없습니다: {stem}")


for stem, roi in CLIPS.items():
    try:
        video_path_of(stem); ok = "✅"
    except FileNotFoundError:
        ok = "❌ 영상 없음"
    print(f"{stem}: {ok} | ROI {'직접 (참조)' if roi else '자동'}")

In [ ]:
# ===== 자동 ROI: 참조 프레임 → 새 프레임으로 경기장 영역 옮기기 =====
# 같은 경기장이면 '경기장 영역을 이미 아는 프레임(참조)'과 새 프레임은 같은 평면(경기장)을 다른 위치·각도에서 본 것.
# 평면 → 평면의 원근 변환은 3×3 호모그래피 H 하나로 표현됨:  [x', y', w']ᵀ = H [x, y, 1]ᵀ,  (x'/w', y'/w')
#   1) SIFT 특징점 추출 (크기·회전·밝기 변화에 강함) — 반 해상도에서 계산해 속도 확보
#   2) 참조 ↔ 새 프레임 특징점 짝짓기 + Lowe 비율 검사(0.75)로 애매한 짝 제거
#   3) RANSAC으로 호모그래피 추정: 무작위 4쌍으로 H를 만들고, 5px 안에 맞는 짝(인라이어)이 가장 많은 H 선택
#      → 나무·자동차처럼 경기장 평면이 아닌 점, 잘못된 짝은 자동으로 버려짐
#   4) 참조 ROI 다각형을 H로 옮김. 참조 화면 가장자리에서 잘린 변은 실제 라인 방향으로 화면 밖까지 연장한 뒤 옮김
#      (새 프레임이 더 넓게 찍혔을 때 ROI가 참조 화면 크기에서 잘리지 않도록)
#   5) 인라이어가 MIN_INLIERS 미만이면 '경기장의 다른 쪽을 찍은 영상'으로 보고 실패 처리 (수동 입력 필요)
import time
SIFT = cv2.SIFT_create(nfeatures=6000)
BF = cv2.BFMatcher()
FEAT_SCALE = 0.5
MIN_INLIERS = 100
RATIO = 0.75


def sift_features(img):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, None, fx=FEAT_SCALE, fy=FEAT_SCALE, interpolation=cv2.INTER_AREA)
    kp, des = SIFT.detectAndCompute(g, None)
    return np.float32([k.pt for k in kp]).reshape(-1, 2) / FEAT_SCALE, des


def _border_sides(p, W, H, tol=3):
    x, y = p
    return {s for s, c in (("L", x <= tol), ("R", x >= W - 1 - tol), ("T", y <= tol), ("B", y >= H - 1 - tol)) if c}


def open_polygon(poly, W, H, L=2500):
    """화면 가장자리에서 잘린 ROI 꼭짓점을 실제 라인 방향으로 L px 연장 (화면 모서리 꼭짓점은 버림)"""
    P = [np.float64(p) for p in poly]; out = []
    for i, p in enumerate(P):
        a, b = P[i - 1], P[(i + 1) % len(P)]
        bp = _border_sides(p, W, H)
        if not bp:
            out.append(p); continue
        ea, eb = bool(bp & _border_sides(a, W, H)), bool(bp & _border_sides(b, W, H))   # 이웃과 같은 가장자리 = 화면 경계 변
        if ea and eb:
            continue
        q = b if ea else a                                                              # 실제 라인 쪽 이웃
        d = (p - q) / (np.linalg.norm(p - q) + 1e-9)
        out.append(p + d * L)
    return np.array(out)


def make_ref(name, img, roi_pts):
    H, W = img.shape[:2]
    pts, des = sift_features(img)
    return {"name": name, "pts": pts, "des": des, "poly_open": open_polygon(roi_pts, W, H), "roi": np.asarray(roi_pts)}


def estimate_transform(ref, pts, des, model="homography"):
    """참조 → 새 프레임 변환과 인라이어 수. model: 'homography'(원근, 8자유도) / 'similarity'(이동·회전·크기, 4자유도)"""
    if des is None or ref["des"] is None or len(des) < 10:
        return None, 0
    good = [p[0] for p in BF.knnMatch(ref["des"], des, k=2) if len(p) == 2 and p[0].distance < RATIO * p[1].distance]
    if len(good) < 10:
        return None, 0
    a = ref["pts"][[m.queryIdx for m in good]]; b = pts[[m.trainIdx for m in good]]
    if model == "homography":
        Hm, inl = cv2.findHomography(a, b, cv2.RANSAC, 5.0)
    else:
        A, inl = cv2.estimateAffinePartial2D(a, b, method=cv2.RANSAC, ransacReprojThreshold=5.0)
        Hm = None if A is None else np.vstack([A, [0, 0, 1]])
    return (None, 0) if Hm is None else (Hm, int(inl.sum()))


def warp_roi_mask(ref, Hm, W, H):
    pts = np.hstack([ref["poly_open"], np.ones((len(ref["poly_open"]), 1))]) @ Hm.T
    if np.any(pts[:, 2] <= 1e-6):                       # 연장한 점이 지평선 너머로 넘어감 → 실패
        return None
    m = np.zeros((H, W), np.uint8)
    cv2.fillPoly(m, [np.round(pts[:, :2] / pts[:, 2:]).astype(np.int32).reshape(-1, 1, 2)], 1)
    return m


def mask_to_poly(m):
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return cv2.approxPolyDP(max(cnts, key=cv2.contourArea), 2.0, True).reshape(-1, 2)


def auto_roi(img, refs, model="homography", exclude=()):
    """→ (ROI 다각형 또는 None, 고른 참조 이름, 인라이어 수)"""
    H, W = img.shape[:2]
    pts, des = sift_features(img)
    best = None
    for ref in refs:
        if ref["name"] in exclude:
            continue
        Hm, n = estimate_transform(ref, pts, des, model)
        if Hm is not None and n >= MIN_INLIERS and (best is None or n > best[2]):
            best = (ref, Hm, n)
    if best is None:
        return None, None, 0
    m = warp_roi_mask(best[0], best[1], W, H)
    if m is None or m.sum() < 0.05 * W * H:             # 화면의 5%도 안 되면 잘못 옮긴 것으로 봄
        return None, best[0]["name"], best[2]
    return mask_to_poly(m), best[0]["name"], best[2]


def expand_polygon(poly, margin, W, H):
    """다각형을 margin(px)만큼 바깥으로 넓힘 (1a와 같은 방법: 채운 마스크 팽창 → 다시 다각형)"""
    mask = np.zeros((H, W), np.uint8)
    cv2.fillPoly(mask, [np.asarray(poly, np.int32).reshape(-1, 1, 2)], 255)
    if margin > 0:
        mask = cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * margin + 1, 2 * margin + 1)))
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return cv2.approxPolyDP(max(cnts, key=cv2.contourArea), 2.0, True).reshape(-1, 2)


def poly_iou(p1, p2, W, H):
    a = np.zeros((H, W), np.uint8); b = np.zeros((H, W), np.uint8)
    cv2.fillPoly(a, [np.int32(p1).reshape(-1, 1, 2)], 1); cv2.fillPoly(b, [np.int32(p2).reshape(-1, 1, 2)], 1)
    return float((a & b).sum() / max(1, (a | b).sum()))

print("✅ 자동 ROI 함수 준비 완료 (OpenCV", cv2.__version__, ")")

In [ ]:
# ② 참조 등록 + 클립별 시작 ROI 미리보기 (초록=직접 ROI, 빨강=자동 ROI)
def inside(poly, x, y):
    return cv2.pointPolygonTest(np.asarray(poly, np.int32).reshape(-1, 1, 2), (float(x), float(y)), False) >= 0


def read_frame(stem, fid=1):
    cap = cv2.VideoCapture(video_path_of(stem)); img = None
    for _ in range(fid):
        ok, img = cap.read()
        assert ok, f"{stem}: {fid}프레임을 읽을 수 없습니다"
    cap.release()
    return img


def draw_grid(vis):
    H, W = vis.shape[:2]
    for x in range(0, W, 100):
        cv2.line(vis, (x, 0), (x, H), (255, 255, 0) if x % 500 else (0, 255, 255), 1)
        cv2.putText(vis, str(x), (x + 2, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    for y in range(0, H, 100):
        cv2.line(vis, (0, y), (W, y), (255, 255, 0) if y % 500 else (0, 255, 255), 1)
        cv2.putText(vis, str(y), (2, y - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    return vis


REFS = []
for stem, pts in CLIPS.items():
    if pts:
        try:
            REFS.append(make_ref(stem, read_frame(stem, REF_FRAME.get(stem, 1)), pts))
        except FileNotFoundError:
            print(f"⚠️ 참조 영상 없음: {stem}")
print("참조:", [r["name"][-6:-2] for r in REFS])

ROI, ROI_INFO = {}, {}
for stem, pts in CLIPS.items():
    try:
        img = read_frame(stem, 1)
    except FileNotFoundError as e:
        print("⏭️", e); continue
    H, W = img.shape[:2]
    vis = img.copy()
    if pts:
        ROI[stem] = expand_polygon(pts, ROI_MARGIN_PX, W, H); ROI_INFO[stem] = "직접"
        color, title = (0, 255, 0), "직접 정한 ROI (참조)"
    else:
        poly, ref, n = auto_roi(img, REFS)
        if poly is not None:
            ROI[stem] = expand_polygon(poly, ROI_MARGIN_PX, W, H); ROI_INFO[stem] = f"자동 (참조 {ref[-6:-2]}, 인라이어 {n})"
            color, title = (0, 0, 255), f"자동 ROI · 참조 {ref[-6:-2]} · 인라이어 {n}"
        else:
            draw_grid(vis); color = None
            title = "자동 ROI 실패: 격자를 보고 꼭짓점을 ①의 CLIPS에 넣어주세요 (이 클립은 건너뜀)"
    if color:
        cv2.polylines(vis, [np.int32(ROI[stem]).reshape(-1, 1, 2)], True, color, 4)
    plt.figure(figsize=(14, 8)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis("off")
    plt.title(f"{stem} · {title}"); plt.show()
print("처리할 클립:", {s[-6:-2]: v for s, v in ROI_INFO.items()})

## ③ ByteTrack (2a·2b와 같은 numpy 구현)

In [ ]:
# ===== ByteTrack (논문 · Ultralytics 구현과 같은 규칙을 그대로 옮긴 버전) =====
# 저장해둔 탐지 결과에 바로 돌릴 수 있도록 numpy만으로 작성했습니다.
#   1) 칼만 필터로 기존 트랙의 이번 위치 예측
#   2) 높은 점수 박스와 IoU 짝짓기 (헝가리안)
#   3) 짝 못 찾은 트랙은 낮은 점수 박스와 한 번 더 짝짓기  ← ByteTrack의 핵심
#   4) 남은 높은 점수 박스 중 new_track_thresh 이상만 새 트랙으로 등록
#   5) track_buffer 프레임 동안 못 찾은 트랙은 삭제

class KalmanXYAH:
    """상태: [cx, cy, 종횡비 a, 높이 h, 각 속도 4개]. 등속 모델."""
    def __init__(self):
        self.F = np.eye(8)
        for i in range(4):
            self.F[i, 4 + i] = 1.0
        self.H = np.eye(4, 8)
        self.wp, self.wv = 1.0 / 20, 1.0 / 160

    def initiate(self, z):
        h = z[3]
        mean = np.r_[z, np.zeros(4)]
        std = [2 * self.wp * h, 2 * self.wp * h, 1e-2, 2 * self.wp * h,
               10 * self.wv * h, 10 * self.wv * h, 1e-5, 10 * self.wv * h]
        return mean, np.diag(np.square(std))

    def predict(self, mean, cov):
        h = mean[3]
        q = [self.wp * h, self.wp * h, 1e-2, self.wp * h, self.wv * h, self.wv * h, 1e-5, self.wv * h]
        return self.F @ mean, self.F @ cov @ self.F.T + np.diag(np.square(q))

    def update(self, mean, cov, z):
        h = mean[3]
        R = np.diag(np.square([self.wp * h, self.wp * h, 1e-1, self.wp * h]))
        S = self.H @ cov @ self.H.T + R
        K = np.linalg.solve(S, (cov @ self.H.T).T).T
        mean = mean + K @ (z - self.H @ mean)
        cov = cov - K @ S @ K.T
        return mean, cov


def xyxy_to_xyah(b):
    w, h = b[2] - b[0], b[3] - b[1]
    return np.array([(b[0] + b[2]) / 2, (b[1] + b[3]) / 2, w / max(h, 1e-6), h], float)


def xyah_to_xyxy(m):
    cx, cy, a, h = m[:4]
    w = a * h
    return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], float)


def iou_matrix_np(a, b):
    a = np.asarray(a, float).reshape(-1, 4); b = np.asarray(b, float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None); inter = wh[..., 0] * wh[..., 1]
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def linear_assign(cost, thresh):
    if cost.size == 0:
        return [], list(range(cost.shape[0])), list(range(cost.shape[1]))
    c = np.where(cost <= thresh, cost, 1e6)
    r, k = linear_sum_assignment(c)
    m = [(i, j) for i, j in zip(r, k) if c[i, j] <= thresh]
    mi = {i for i, _ in m}; mj = {j for _, j in m}
    return m, [i for i in range(cost.shape[0]) if i not in mi], [j for j in range(cost.shape[1]) if j not in mj]


class Track:
    _next_id = 0
    TRACKED, LOST, REMOVED = 0, 1, 2

    def __init__(self, box, score, det_idx, kf):
        self.kf, self.box, self.score, self.det_idx = kf, np.asarray(box, float), score, det_idx
        self.mean = self.cov = None
        self.state, self.activated, self.track_id = Track.TRACKED, False, -1
        self.start_frame = self.end_frame = 0

    def predicted_box(self):
        return xyah_to_xyxy(self.mean) if self.mean is not None else self.box

    def predict(self):
        m = self.mean.copy()
        if self.state != Track.TRACKED:
            m[7] = 0.0                          # 놓친 트랙은 높이 변화 속도 0
        self.mean, self.cov = self.kf.predict(m, self.cov)

    def activate(self, frame_id):
        Track._next_id += 1
        self.track_id = Track._next_id
        self.mean, self.cov = self.kf.initiate(xyxy_to_xyah(self.box))
        self.state = Track.TRACKED
        self.activated = frame_id == 1
        self.start_frame = self.end_frame = frame_id

    def update(self, det, frame_id):
        self.mean, self.cov = self.kf.update(self.mean, self.cov, xyxy_to_xyah(det.box))
        self.box, self.score, self.det_idx = det.box, det.score, det.det_idx
        self.state, self.activated, self.end_frame = Track.TRACKED, True, frame_id


class ByteTracker:
    def __init__(self, track_high_thresh=0.5, track_low_thresh=0.1, new_track_thresh=0.6,
                 track_buffer=30, match_thresh=0.8, fuse_score=True, frame_rate=30):
        self.hi, self.lo, self.new = track_high_thresh, track_low_thresh, new_track_thresh
        self.max_lost = int(frame_rate / 30.0 * track_buffer)
        self.match, self.fuse = match_thresh, fuse_score
        self.kf = KalmanXYAH()
        self.tracked, self.lost = [], []
        self.frame_id = 0
        Track._next_id = 0

    def _dist(self, tracks, dets, fuse):
        d = 1 - iou_matrix_np([t.predicted_box() for t in tracks], [x.box for x in dets])
        if fuse and len(dets):
            d = 1 - (1 - d) * np.array([x.score for x in dets])[None, :]
        return d

    def update(self, boxes, scores):
        """boxes (N,4) xyxy, scores (N,) → [(track_id, det_idx), ...] 이번 프레임에 확정된 트랙"""
        self.frame_id += 1
        f = self.frame_id
        boxes = np.asarray(boxes, float).reshape(-1, 4); scores = np.asarray(scores, float).reshape(-1)
        high = [Track(boxes[i], scores[i], i, self.kf) for i in np.where(scores >= self.hi)[0]]
        low = [Track(boxes[i], scores[i], i, self.kf) for i in np.where((scores > self.lo) & (scores < self.hi))[0]]

        unconfirmed = [t for t in self.tracked if not t.activated]
        confirmed = [t for t in self.tracked if t.activated]
        pool = confirmed + [t for t in self.lost if t not in confirmed]
        for t in pool:
            t.predict()

        refound, activated_now = [], []
        # 1차: 높은 점수 박스
        m, u_t, u_d = linear_assign(self._dist(pool, high, self.fuse), self.match)
        for i, j in m:
            t = pool[i]
            if t.state != Track.TRACKED:
                refound.append(t)
            t.update(high[j], f); activated_now.append(t)
        # 2차: 남은 '추적 중' 트랙 × 낮은 점수 박스
        r_tracked = [pool[i] for i in u_t if pool[i].state == Track.TRACKED]
        m2, u_t2, _ = linear_assign(self._dist(r_tracked, low, False), 0.5)
        for i, j in m2:
            r_tracked[i].update(low[j], f); activated_now.append(r_tracked[i])
        for i in u_t2:
            r_tracked[i].state = Track.LOST
        # 확정 안 된(1프레임짜리) 트랙 × 남은 높은 점수 박스
        rest_high = [high[j] for j in u_d]
        m3, u_un, u_d3 = linear_assign(self._dist(unconfirmed, rest_high, self.fuse), 0.7)
        for i, j in m3:
            unconfirmed[i].update(rest_high[j], f); activated_now.append(unconfirmed[i])
        for i in u_un:
            unconfirmed[i].state = Track.REMOVED
        # 새 트랙
        new_tracks = []
        for j in u_d3:
            d = rest_high[j]
            if d.score >= self.new:
                d.activate(f); new_tracks.append(d)
        # 오래 못 찾은 트랙 삭제
        for t in pool:
            if t.state == Track.LOST and f - t.end_frame > self.max_lost:
                t.state = Track.REMOVED

        everyone = {id(t): t for t in self.tracked + self.lost + new_tracks}
        self.tracked = [t for t in everyone.values() if t.state == Track.TRACKED]
        self.lost = [t for t in everyone.values() if t.state == Track.LOST]
        return [(t.track_id, t.det_idx) for t in self.tracked if t.activated and t.end_frame == f]


def _bytetrack_selftest():
    """작은 합성 장면으로 동작 확인: 점수가 잠깐 낮아지거나 몇 프레임 사라져도 ID 유지, 약한 새 박스는 새 ID 안 받음"""
    tr = ByteTracker(track_high_thresh=0.5, track_low_thresh=0.1, new_track_thresh=0.6, track_buffer=30)
    ids = defaultdict(set)
    for f in range(1, 61):
        boxes, scores = [], []
        x = 100 + 3 * f                                     # 선수 A: 오른쪽으로 이동
        if not (30 <= f < 36):                              # 30~35프레임 동안 사라짐(가려짐)
            boxes.append([x, 100, x + 12, 130]); scores.append(0.3 if 15 <= f < 20 else 0.8)   # 15~19 낮은 점수
        boxes.append([300, 200 + f, 312, 230 + f]); scores.append(0.9)                      # 선수 B
        if f == 40:
            boxes.append([500, 50, 512, 80]); scores.append(0.4)                            # 약한 오탐 1번
        for tid, di in tr.update(boxes, scores):
            ids["A" if boxes[di][0] < 290 else ("B" if boxes[di][0] < 400 else "X")].add(tid)
    assert len(ids["A"]) == 1 and len(ids["B"]) == 1 and "X" not in ids, dict(ids)
    print("✅ ByteTrack 자체 점검 통과:", {k: sorted(v) for k, v in ids.items()})

_bytetrack_selftest()

In [ ]:
# ④ 모델 불러오기: 파인튜닝 YOLO(탐지 + 팀) + 추적기 설정 + 화면 이동 검사
import torch
from ultralytics import YOLO
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("장치:", "GPU" if DEVICE == 0 else "CPU (매우 느려요: 런타임 → 런타임 유형 변경 → T4 GPU)")

det_model = YOLO(DET_WEIGHTS)
TEAM = [det_model.names[i] for i in range(len(det_model.names))]
TRACK_PARAMS = json.load(open(TRACKER_CFG_PATH, encoding="utf-8"))["params"]
print("클래스:", TEAM)
print("추적기 설정:", TRACK_PARAMS)

ORB = cv2.ORB_create(nfeatures=1500)
ORB_BF = cv2.BFMatcher(cv2.NORM_HAMMING)     # BF(SIFT용)와 이름을 따로 둠


def motion_feat(img):
    g = cv2.cvtColor(cv2.resize(img, None, fx=MOTION_SCALE, fy=MOTION_SCALE, interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY)
    return ORB.detectAndCompute(g, None)


def motion_iou(fa, fb, roi, W, H):
    """기준 프레임 → 지금 프레임의 화면 이동(호모그래피)으로 ROI를 옮겼을 때 원래 ROI와의 IoU. 측정 실패 = 0 (= 재계산)"""
    (ka, da), (kb, db) = fa, fb
    if da is None or db is None or len(ka) < 30 or len(kb) < 30:
        return 0.0
    good = [p[0] for p in ORB_BF.knnMatch(da, db, k=2) if len(p) == 2 and p[0].distance < 0.75 * p[1].distance]
    if len(good) < 30:
        return 0.0
    src = np.float32([ka[m.queryIdx].pt for m in good]) / MOTION_SCALE
    dst = np.float32([kb[m.trainIdx].pt for m in good]) / MOTION_SCALE
    Hm, inl = cv2.findHomography(src, dst, cv2.RANSAC, 3.0 / MOTION_SCALE)
    if Hm is None or inl is None or inl.sum() < 20:
        return 0.0
    moved = cv2.perspectiveTransform(np.float32(roi).reshape(-1, 1, 2), Hm).reshape(-1, 2)
    return poly_iou(moved, roi, W, H)

In [ ]:
# ⑤ 클립 처리 함수
def label_window(rows, k):
    """트랙 안에서 앞뒤 k프레임의 YOLO 클래스를 점수 가중 다수결 (6a)"""
    T = defaultdict(list)
    for fid in sorted(rows):
        for tid, b, s, c in rows[fid]:
            T[tid].append((fid, c, s))
    lab = {}
    for tid, seq in T.items():
        f = np.array([x[0] for x in seq]); c = np.array([x[1] for x in seq]); s = np.array([x[2] for x in seq])
        lo = np.searchsorted(f, f - k, "left"); hi = np.searchsorted(f, f + k, "right")
        for i in range(len(seq)):
            w = np.bincount(c[lo[i]:hi[i]], weights=s[lo[i]:hi[i]], minlength=len(TEAM))
            lab[(tid, int(f[i]))] = int(w.argmax())
    return lab, T


def split_on_team_change(rows, lab, T):
    """라벨이 바뀌는 지점에서 트랙을 새 ID로 자름 (다른 팀 선수에게 넘어간 ID 분리)"""
    seg_of = {}
    for tid, seq in T.items():
        seg, prev = 0, None
        for fid, _, _ in seq:
            l = lab[(tid, fid)]
            if prev is not None and l != prev:
                seg += 1
            prev = l; seg_of[(tid, fid)] = seg
    new_id, new_rows, new_lab = {}, {}, {}
    for fid in sorted(rows):
        rr = []
        for tid, b, s, c in rows[fid]:
            nid = new_id.setdefault((tid, seg_of[(tid, fid)]), len(new_id) + 1)
            rr.append((nid, b, s, c)); new_lab[(nid, fid)] = lab[(tid, fid)]
        new_rows[fid] = rr
    return new_rows, new_lab


def local_copy(stem):
    """Drive에서 바로 읽으면 느려서 Colab 로컬 디스크로 복사"""
    dst = os.path.join("/content/_scratch_video", os.path.basename(video_path_of(stem)))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        shutil.copy(video_path_of(stem), dst)
    return dst


def process_clip(stem):
    """한 번 읽으면서 ROI 갱신 → 탐지(+팀) → ROI 안만 → 추적. 끝난 뒤 앞뒤 15프레임 팀 라벨 + 트랙 자르기."""
    cap = cv2.VideoCapture(local_copy(stem))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    tracker = ByteTracker(**TRACK_PARAMS, fuse_score=True, frame_rate=int(round(fps)))
    roi, auto = ROI[stem], (CLIPS.get(stem) is None or ROI_FOLLOW)
    roi_changes, roi_why, roi_fail, ref_feat = {1: np.asarray(roi).tolist()}, Counter(), 0, None
    rows = {}
    fid, t0 = 0, time.time()
    while True:
        ok, img = cap.read()
        if not ok:
            break
        fid += 1                                            # v3와 같게 프레임 번호는 1부터
        H, W = img.shape[:2]
        if auto:
            why = None
            if fid == 1:
                ref_feat = motion_feat(img)                 # 시작 ROI를 정한 화면 = 이동 검사의 기준
            elif (fid - 1) % ROI_UPDATE_EVERY == 0:
                why = "주기"
            elif (fid - 1) % MOTION_EVERY == 0:
                f = motion_feat(img)
                if motion_iou(ref_feat, f, roi, W, H) < MOTION_IOU_MIN:
                    why = "이동"
            if why:
                poly, _, _ = auto_roi(img, REFS)
                if poly is not None:
                    roi = expand_polygon(poly, ROI_MARGIN_PX, W, H); roi_changes[fid] = np.asarray(roi).tolist()
                    ref_feat = motion_feat(img); roi_why[why] += 1
                else:
                    roi_fail += 1                           # 실패하면 직전 ROI 유지
        r = det_model.predict(img, imgsz=DET_IMGSZ, conf=DET_CONF, max_det=300, device=DEVICE, verbose=False)[0]
        bx = r.boxes.xyxy.cpu().numpy().reshape(-1, 4); sc = r.boxes.conf.cpu().numpy().reshape(-1)
        cl = r.boxes.cls.cpu().numpy().astype(int).reshape(-1)
        keep = [i for i, b in enumerate(bx) if inside(roi, (b[0] + b[2]) / 2, b[3])]   # 발 위치가 ROI 안
        bx, sc, cl = bx[keep], sc[keep], cl[keep]
        rows[fid] = [(int(t), bx[d], float(sc[d]), int(cl[d])) for t, d in tracker.update(bx, sc)]
        if fid % 300 == 0:
            el = time.time() - t0
            print(f"  [{stem}] {fid}/{n_total}프레임 · {el:.0f}s 경과 · 남은 시간 약 {el / fid * max(0, n_total - fid) / 60:.1f}분")
    cap.release()
    lab, T = label_window(rows, TEAM_WINDOW)
    n_raw = len(T)
    if SPLIT_ON_TEAM_CHANGE:
        rows, lab = split_on_team_change(rows, lab, T)
    if auto:
        print(f"  자동 ROI 갱신: 주기 {roi_why['주기']}번 · 이동 감지 {roi_why['이동']}번 · 실패(직전 ROI 유지) {roi_fail}번")
    return {"fps": fps, "n_frames": fid, "roi_changes": roi_changes, "roi_why": dict(roi_why), "rows": rows, "lab": lab,
            "n_tracks_raw": n_raw, "seconds": time.time() - t0}


def save_json(stem, res):
    frames = []
    for fid in sorted(res["rows"]):
        players = []
        for tid, box, s, c in res["rows"][fid]:
            x1, y1, x2, y2 = [int(round(v)) for v in box]
            players.append({"track_id": tid, "type": TEAM[res["lab"][(tid, fid)]], "bbox": [x1, y1, x2, y2],
                            "feet": [(x1 + x2) // 2, y2], "conf": round(s, 4), "yolo_cls": TEAM[c]})
        frames.append({"frame_id": int(fid), "players": players})
    data = {"video_name": os.path.basename(video_path_of(stem)), "fps": res["fps"],
            "field_polygon": np.asarray(ROI[stem]).tolist(), "roi_mode": ROI_INFO[stem], "frames": frames,
            "notes": [f"v7: 파인튜닝 YOLO26m imgsz{DET_IMGSZ} conf{DET_CONF} + ROI(여유 {ROI_MARGIN_PX}px, {ROI_UPDATE_EVERY}프레임마다 + 이동 감지) "
                      f"+ ByteTrack {TRACK_PARAMS} + 팀 라벨 앞뒤 {TEAM_WINDOW}프레임 다수결" + (" + 팀 바뀌면 트랙 자르기" if SPLIT_ON_TEAM_CHANGE else ""),
                      "type = 앞뒤 15프레임 다수결로 정한 팀 · yolo_cls = 그 프레임에서 YOLO가 고른 클래스"]}
    if len(res["roi_changes"]) > 1:
        data["field_polygon_by_frame"] = {str(k): v for k, v in res["roi_changes"].items()}   # 바뀐 프레임부터 적용
    p = os.path.join(OUT_DIR, f"{stem}_tracking_final.json")
    with open(p, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    return p


COLORS = {"Hongik": (255, 140, 0), "Kookmin": (40, 40, 230), "Referee": (0, 215, 255), "Unknown": (160, 160, 160)}  # BGR: 파랑·빨강·노랑·회색


def draw(img, players, poly):
    cv2.polylines(img, [np.asarray(poly, np.int32).reshape(-1, 1, 2)], True, (0, 255, 0), 2)
    for p in players:
        x1, y1, x2, y2 = p["bbox"]; c = COLORS.get(p["type"], (160, 160, 160))
        cv2.rectangle(img, (x1, y1), (x2, y2), c, 2)
        cv2.putText(img, f"{p['type'][0]}{p['track_id']}", (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, c, 1, cv2.LINE_AA)
    return img


def poly_at(data, fid):
    ch = data.get("field_polygon_by_frame")
    if not ch:
        return data["field_polygon"]
    k = max((int(f) for f in ch if int(f) <= fid), default=None)
    return ch[str(k)] if k is not None else data["field_polygon"]


def render_video(stem, json_path):
    data = json.load(open(json_path, encoding="utf-8"))
    by_f = {fr["frame_id"]: fr["players"] for fr in data["frames"]}
    cap = cv2.VideoCapture(local_copy(stem))
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    tmp = f"/content/{stem}_v7.mp4"
    vw = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*"mp4v"), data["fps"], (W, H))
    fid = 0
    while True:
        ok, img = cap.read()
        if not ok:
            break
        fid += 1
        img = draw(img, by_f.get(fid, []), poly_at(data, fid))
        cv2.putText(img, f"frame {fid}", (20, H - 20), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
        vw.write(img)
    cap.release(); vw.release()
    dst = os.path.join(OUT_DIR, f"{stem}_v7.mp4")
    shutil.copy(tmp, dst)
    return dst

In [ ]:
# ⑥ 실행
SUMMARY = []
for stem in ROI:
    jp = os.path.join(OUT_DIR, f"{stem}_tracking_final.json")
    if os.path.exists(jp) and not OVERWRITE:
        print(f"⏭️ {stem}: 결과가 이미 있어요 (다시 하려면 OVERWRITE = True)")
    else:
        print(f"▶ {stem} 처리 시작")
        res = process_clip(stem)
        jp = save_json(stem, res)
        ids = {(t, TEAM[res["lab"][(t, f)]]) for f, rr in res["rows"].items() for t, b, s, c in rr}
        cnt = Counter(team for _, team in ids)
        SUMMARY.append({"클립": stem, "ROI": ROI_INFO[stem] + (" + 따라감" if ROI_FOLLOW and CLIPS.get(stem) else ""), "프레임": res["n_frames"],
                        "ROI 갱신 (주기/이동)": f"{res['roi_why'].get('주기', 0)}/{res['roi_why'].get('이동', 0)}",
                        "추적 ID (자르기 전→후)": f"{res['n_tracks_raw']}→{len(ids)}",
                        "홍익 ID": cnt["Hongik"], "국민 ID": cnt["Kookmin"], "심판 ID": cnt["Referee"],
                        "처리 시간(분)": round(res["seconds"] / 60, 1)})
        print(f"✅ JSON 저장 → {jp}")
    if MAKE_VIDEO and (OVERWRITE or not os.path.exists(os.path.join(OUT_DIR, f"{stem}_v7.mp4"))):
        print(f"🎬 확인용 영상 저장 → {render_video(stem, jp)}")
if SUMMARY:
    display(pd.DataFrame(SUMMARY))

In [ ]:
# ⑦ 결과 확인: 클립마다 3장 (처음 · 가운데 · 끝 근처), 프레임당 팀별 인원
for stem in ROI:
    data = json.load(open(os.path.join(OUT_DIR, f"{stem}_tracking_final.json"), encoding="utf-8"))
    by_f = {fr["frame_id"]: fr["players"] for fr in data["frames"]}
    n = len(data["frames"])
    picks = [max(1, int(n * q)) for q in (0.1, 0.5, 0.9)]
    cap = cv2.VideoCapture(local_copy(stem)); imgs, fid = {}, 0
    while fid < max(picks):
        ok, img = cap.read()
        if not ok:
            break
        fid += 1
        if fid in picks:
            imgs[fid] = draw(img, by_f.get(fid, []), poly_at(data, fid))
    cap.release()
    fig, axes = plt.subplots(1, len(imgs), figsize=(6 * len(imgs), 4))
    for ax, (f, im) in zip(np.atleast_1d(axes), imgs.items()):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(f"{stem[-6:-2]} · frame {f}"); ax.axis("off")
    plt.tight_layout(); plt.show()
    per = pd.DataFrame([Counter(p["type"] for p in fr["players"]) for fr in data["frames"]],
                       index=[fr["frame_id"] for fr in data["frames"]]).fillna(0)
    over = int((per.reindex(columns=["Hongik", "Kookmin"]).fillna(0).max(axis=1) > 14).sum())
    print(f"{stem}: 프레임당 평균 인원", per.mean().round(1).to_dict(), f"· 한 팀 14명 초과 프레임 {over}개")
    plt.figure(figsize=(14, 2.6))
    for team, c in [("Hongik", "tab:blue"), ("Kookmin", "tab:red"), ("Referee", "goldenrod")]:
        if team in per:
            plt.plot(per.index, per[team].rolling(15, min_periods=1).mean(), color=c, label=team)
    plt.axhline(11, color="gray", ls="--", lw=1); plt.ylabel("인원 (15프레임 평균)"); plt.xlabel("frame")
    plt.title(f"{stem[-6:-2]} · 프레임별 팀 인원 (점선 = 11명)"); plt.legend(loc="upper right"); plt.grid(alpha=0.3); plt.show()

In [ ]:
# ⑧ (선택) 실험 결과와 일치하는지 점검: 평가 클립은 6b의 B_motion 결과(득점 구간)와 박스 · 팀이 거의 같아야 해요
def compare_with(stem, ref_dir=os.path.join(EVAL_DIR, "runs", "v6b_roi_B_motion")):
    rp = os.path.join(ref_dir, f"{stem}_tracking_final.json")
    if not os.path.exists(rp):
        return None
    a = {fr["frame_id"]: fr["players"] for fr in json.load(open(os.path.join(OUT_DIR, f"{stem}_tracking_final.json"), encoding="utf-8"))["frames"]}
    b = {fr["frame_id"]: fr["players"] for fr in json.load(open(rp, encoding="utf-8"))["frames"]}
    n_a = n_b = same_box = same_lab = 0
    for f in set(a) & set(b):
        n_a += len(a[f]); n_b += len(b[f])
        kb = {tuple(p["bbox"]): p["type"] for p in b[f]}
        for p in a[f]:
            t = kb.get(tuple(p["bbox"]))
            if t is not None:
                same_box += 1; same_lab += int(t == p["type"])
    return {"클립": stem, "박스 수 (실행/실험)": f"{n_a}/{n_b}", "같은 박스 비율": round(same_box / max(1, n_b), 3),
            "같은 박스 중 팀 일치": round(same_lab / max(1, same_box), 3)}

rows = [r for r in (compare_with(s) for s in ROI) if r]
if rows:
    display(pd.DataFrame(rows))
    print("0.95 이상이면 실험(6b)과 같은 결과를 내는 것. 실험은 캐시된 탐지를 썼고 여기선 다시 탐지해서,")
    print("GPU 연산 차이로 박스 좌표가 1px씩 다를 수 있고 시작 ROI도 조금 달라 100%는 아닐 수 있어요.")
else:
    print("비교할 실험 결과가 없어요 (평가 클립이 아니면 정상)")